### Imports

In [2]:
# !java -version

In [ ]:
# %pip install pyspark graphframes-py==0.10.0 graphframes-py==0.9.0 

In [ ]:
# %pip install --upgrade grpcio-status>=1.48.1 grpcio google protobuf pyarrow

In [5]:
# !pip list | grep pyspark
# !pip list | grep graphframes

In [7]:
# !pip list | findstr pyspark
# !pip list | findstr graphframes
# !pyspark --version

In [4]:
# !pip install findspark networkx matplotlib pyvis

In [8]:
# import os

# os.environ["HADOOP_HOME"] = r"C:\hadoop\hadoop-3.3.6"
# os.environ["PATH"] += os.pathsep + r"C:\hadoop\hadoop-3.3.6\bin"

# print(os.environ.get("HADOOP_HOME"))
# print(os.environ.get("PATH"))

In [10]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import * 
from pyspark.sql.types import *

In [ ]:
# spark = SparkSession.builder \
#     .appName("GraphProject") \
#     .config("spark.jars.packages", "io.graphframes:graphframes-spark3_2.13:0.9.0-spark3.5") \
#     .getOrCreate()

In [11]:
spark = (
    SparkSession.builder
    .appName("AmazonGraph")
    .config("spark.jars.packages", "io.graphframes:graphframes-spark3_2.12:0.10.0")
    .getOrCreate()
)

print("Active Spark sessions:", spark.sparkContext.uiWebUrl)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/11/21 21:33:25 WARN Utils: Your hostname, Kenuey, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/11/21 21:33:25 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/kenuey/.local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/kenuey/.ivy2.5.2/cache
The jars for the packages stored in: /home/kenuey/.ivy2.5.2/jars
io.graphframes#graphframes-spark3_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-3d3d9b19-4706-4b7b-acdc-7fd1b2a8daed;1.0
	confs: [default]
	found io.graphframes#graphframes-spark3_2.12;0.10.0 in central
	found io.graphframes#graphframes-graphx-spark3_2.12;0.10.0 in central
:: resolution report :: resolve 155ms :: artifacts dl 6ms
	:: modules in use:
	io.graph

Active Spark sessions: http://10.255.255.254:4040


In [30]:
AMAZONMETA_TXT = 'data/amazon-meta.txt'
RECOMMS_CSV = 'data/recomms.csv'
MEASURES_JSON = 'data/measures.json'

In [32]:
raw = spark.read.text(AMAZONMETA_TXT)

raw.show(20)

+--------------------+
|               value|
+--------------------+
|# Full informatio...|
| Total items: 548552|
|                    |
|             Id:   0|
|    ASIN: 0771044445|
|  discontinued pr...|
|                    |
|             Id:   1|
|    ASIN: 0827229534|
|  title: Patterns...|
|         group: Book|
|   salesrank: 396585|
|  similar: 5  080...|
|       categories: 2|
|   |Books[283155]...|
|   |Books[283155]...|
|  reviews: total:...|
|    2000-7-28  cu...|
|    2003-12-14  c...|
|                    |
+--------------------+
only showing top 20 rows


In [58]:
rdd = spark.sparkContext.textFile(AMAZONMETA_TXT)

### Data Preprocessing

In [59]:
from pyspark.sql.types import *
from pyspark.sql import Row

# 1. Разбиваем на блоки
def split_blocks(iter):
    block = []
    for line in iter:
        line = line.strip()
        if line.startswith("Id:") and block:
            yield block
            block = []
        block.append(line)
    if block:
        yield block

In [60]:
# 2. Парсим блок продукта
def parse_block(block):
    product = {
        "id": None,
        "asin": None,
        "title": "",
        "group": None,
        "salesrank": None,
        "similar_count": 0,
        "similar_asins": []
    }

    for line in block:
        if line.startswith("Id:"):
            product["id"] = line.split()[1]
        elif line.startswith("ASIN:"):
            product["asin"] = line.split()[1]
        elif line.startswith("title:"):
            product["title"] = line.replace("title:", "").strip()
        elif line.startswith("group:"):
            product["group"] = line.split(":")[1].strip()
        elif line.startswith("salesrank:"):
            try:
                product["salesrank"] = int(line.split(":")[1].strip())
            except:
                product["salesrank"] = None
        elif line.startswith("similar:"):
            parts = line.split()
            product["similar_count"] = int(parts[1])
            product["similar_asins"] = parts[2:]

    return [product]

In [61]:
# 3. rdd → blocks → parsed dicts
blocks_rdd = rdd.mapPartitions(split_blocks)
parsed_rdd = blocks_rdd.flatMap(parse_block)

In [62]:
# 4. схема
schema = StructType([
    StructField("id", StringType(), True),
    StructField("asin", StringType(), True),
    StructField("title", StringType(), True),
    StructField("group", StringType(), True),
    StructField("salesrank", IntegerType(), True),
    StructField("similar_count", IntegerType(), True),
    StructField("similar_asins", ArrayType(StringType()), True)
])

In [63]:
products_df = spark.createDataFrame(parsed_rdd, schema)
products_df.show(10, truncate=False)

+----+----------+-----------------------------------------------------------------+-----+---------+-------------+------------------------------------------------------------+
|id  |asin      |title                                                            |group|salesrank|similar_count|similar_asins                                               |
+----+----------+-----------------------------------------------------------------+-----+---------+-------------+------------------------------------------------------------+
|NULL|NULL      |                                                                 |NULL |NULL     |0            |[]                                                          |
|0   |0771044445|                                                                 |NULL |NULL     |0            |[]                                                          |
|1   |0827229534|Patterns of Preaching: A Sermon Sampler                          |Book |396585   |5            |[0804215715,

In [64]:
vertices = products_df.select(
    products_df.asin.alias("id"),
    "title",
    "group",
    "salesrank"
).distinct()


In [65]:
from pyspark.sql.functions import explode, col

edges = products_df \
    .withColumn("similar", explode("similar_asins")) \
    .select(
        col("asin").alias("src"),
        col("similar").alias("dst")
    )


In [66]:
from graphframes import GraphFrame

g = GraphFrame(vertices, edges)


In [67]:
# Vertices
vertices = products_df.selectExpr("asin as id", "title")

# Edges
from pyspark.sql.functions import explode, col

edges = products_df \
    .withColumn("dst", explode(col("similar_asins"))) \
    .select(col("asin").alias("src"), col("dst"))

edges.show(5)


+----------+----------+
|       src|       dst|
+----------+----------+
|0827229534|0804215715|
|0827229534|156101074X|
|0827229534|0687023955|
|0827229534|0687074231|
|0827229534|082721619X|
+----------+----------+
only showing top 5 rows


In [ ]:
vertices = products_df.selectExpr("asin as id", "title", "group", "salesrank")


edges = products_df.rdd.flatMap(lambda row: [
    Row(src=row.asin, dst=asin) for asin in row.similar_asins
]).toDF()


In [68]:
g = GraphFrame(vertices, edges)

# Пример: показать вершины и ребра
print("Vertices:")
g.vertices.show(5, truncate=False)

print("Edges:")
g.edges.show(5, truncate=False)


Vertices:
+----------+------------------------------------------------+
|id        |title                                           |
+----------+------------------------------------------------+
|NULL      |                                                |
|0771044445|                                                |
|0827229534|Patterns of Preaching: A Sermon Sampler         |
|0738700797|Candlemas: Feast of Flames                      |
|0486287785|World War II Allied Fighter Planes Trading Cards|
+----------+------------------------------------------------+
only showing top 5 rows
Edges:
+----------+----------+
|src       |dst       |
+----------+----------+
|0827229534|0804215715|
|0827229534|156101074X|
|0827229534|0687023955|
|0827229534|0687074231|
|0827229534|082721619X|
+----------+----------+
only showing top 5 rows


In [ ]:
# Пример: количество соседей для каждого товара
g.degrees.show(5)

# PageRank (важность товара в графе)
results = g.pageRank(resetProbability=0.15, maxIter=5)
results.vertices.select("id", "pagerank").show(5)


### Descriptive Analysis

### Bundles and Collections

### Graph Visualization

### New Recommender System